In [11]:
"""

Raw data format (tab-separated .txt, no header):
    frame_id    track_id    pos_x    pos_y

Frames are spaced by 10 (i.e., 0, 10, 20, ...) which corresponds to
0.4 seconds between consecutive samples.

The 5 scenes are: ETH, HOTEL, UNIV, ZARA1, ZARA2.
Leave-one-out: to evaluate on scene X, train on the other 4.

Each training sample is a 20-step trajectory:
    - 8 observed steps  (3.2 seconds)  →  input to the model
    - 12 predicted steps (4.8 seconds)  →  what the model learns to generate

Samples are extracted with a sliding window over time: at every frame t,
any pedestrian visible continuously from t to t+19 yields one sample.

"""

import os
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

In [12]:
## these are some constants followed in the socialGAN dataset, which are used for tajectory prediction

OBS_LEN = 8          # 8 observed timesteps  = 3.2 seconds
PRED_LEN = 12        # 12 predicted timesteps = 4.8 seconds
SEQ_LEN = OBS_LEN + PRED_LEN   # 20 total
FRAME_SKIP = 10      # raw frames: 0, 10, 20, ... → we normalize to 0, 1, 2, ...
DT = 0.4             # seconds between consecutive (normalized) frames

# there are 5 ETH/UCY scenes and their test files
SCENES = ["eth", "hotel", "univ", "zara1", "zara2"]


def parse_trajectory_file(filepath):
    """
    Read a raw trajectory .txt file.

    Returns a DataFrame with columns: [frame, ped_id, x, y]
    where frame IDs are normalized (divided by FRAME_SKIP).
    """
    df = pd.read_csv(filepath, sep="\t", header=None)
    df.columns = ["frame", "ped_id", "x", "y"]

    ## types
    df["frame"] = df["frame"].astype(int)
    df["ped_id"] = df["ped_id"].astype(int)
    df["x"] = df["x"].astype(float)
    df["y"] = df["y"].astype(float)

    # normalize frame IDs from 0, 10, 20, .. to → 0, 1, 2, ...
    df["frame"] = df["frame"] // FRAME_SKIP

    df = df.sort_values(["frame", "ped_id"]).reset_index(drop=True)
    return df

In [13]:
def extract_trajectories(df, scene_name=""):
    """
    From a DataFrame of (frame, ped_id, x, y), extract all valid
    20-step trajectory windows using a sliding window.

    A valid sample requires a pedestrian to be present at every
    frame in [t, t+1, ..., t+19] with no gaps.

    Returns a list of dicts, each containing:
        "observed":  np.array of shape (8, 2)
        "future":    np.array of shape (12, 2)
        "ped_id":    int
        "scene":     str
        "frame":     int (the starting frame of the window)
    """
    samples = []
    frames = sorted(df["frame"].unique())
    min_frame, max_frame = frames[0], frames[-1]

    # Group data by pedestrian for fast lookup
    ped_groups = {}
    for ped_id, group in df.groupby("ped_id"):
        # Build a dict: frame → (x, y) for this pedestrian
        frame_to_pos = {}
        for _, row in group.iterrows():
            frame_to_pos[row["frame"]] = np.array([row["x"], row["y"]])
        ped_groups[ped_id] = frame_to_pos

    # Slide the window across all possible starting frames
    for t_start in range(min_frame, max_frame - SEQ_LEN + 2):
        needed_frames = list(range(t_start, t_start + SEQ_LEN))

        for ped_id, frame_to_pos in ped_groups.items():
            # Check if this pedestrian is present at ALL 20 frames
            if all(f in frame_to_pos for f in needed_frames):
                trajectory = np.array([frame_to_pos[f] for f in needed_frames])

                samples.append({
                    "observed": trajectory[:OBS_LEN],     # (8, 2)
                    "future":   trajectory[OBS_LEN:],     # (12, 2)
                    "ped_id":   ped_id,
                    "scene":    scene_name,
                    "frame":    t_start,
                })

    return samples

In [14]:
class ETHUCYDataset(Dataset):
    """
    PyTorch Dataset for ETH/UCY trajectory prediction.

    Each sample returns:
        observed: (8, 2) float tensor  — past positions
        future:   (12, 2) float tensor — future positions to predict

    Optionally applies rotation augmentation for training.
    """

    def __init__(self, samples, augment=False):
        """
        Args:
            samples: list of dicts from extract_trajectories()
            augment: if True, apply random rotation augmentation
        """
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        observed = sample["observed"].copy()   # (8, 2)
        future = sample["future"].copy()       # (12, 2)

        if self.augment:
            angle = np.random.uniform(0, 2 * np.pi)
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            rot = np.array([[cos_a, -sin_a],
                            [sin_a,  cos_a]])
            observed = observed @ rot.T
            future = future @ rot.T

        return {
            "observed": torch.tensor(observed, dtype=torch.float32),
            "future":   torch.tensor(future, dtype=torch.float32),
            "ped_id":   sample["ped_id"],
            "scene":    sample["scene"],
            "frame":    sample["frame"],
        }

In [15]:
def get_ethucy_split(scene_name, split, data_root="./raw_data", augment=None):
    """
    Load the ETH/UCY data for a given scene and split.

    Args:
        scene_name: one of "eth", "hotel", "univ", "zara1", "zara2"
        split:      "train", "val", or "test"
        data_root:  path to directory containing eth/, hotel/, etc.
        augment:    whether to apply rotation augmentation
                    (defaults to True for train, False otherwise)

    Returns:
        ETHUCYDataset instance
    """
    assert scene_name in SCENES, f"Unknown scene: {scene_name}. Use one of {SCENES}"
    assert split in ("train", "val", "test"), f"Split must be train/val/test"

    if augment is None:
        augment = (split == "train")

    split_dir = os.path.join(data_root, scene_name, split)
    txt_files = sorted(glob.glob(os.path.join(split_dir, "*.txt")))

    if not txt_files:
        raise FileNotFoundError(
            f"No .txt files found in {split_dir}. "
            f"Make sure the raw data is at {data_root}/"
        )

    all_samples = []
    for fpath in txt_files:
        fname = os.path.basename(fpath)
        df = parse_trajectory_file(fpath)
        samples = extract_trajectories(df, scene_name=fname)
        print(f"  {fname}: {len(samples)} trajectory samples "
              f"({df['ped_id'].nunique()} pedestrians, "
              f"{df['frame'].nunique()} frames)")
        all_samples.extend(samples)

    print(f"  Total: {len(all_samples)} samples for {scene_name}/{split}\n")
    return ETHUCYDataset(all_samples, augment=augment)

In [18]:
if __name__ == "__main__":

    DATA_ROOT = "./raw_data"

    print("=" * 60)
    print("ETH/UCY Dataset Loader")
    print("=" * 60)

    # ── Load all 5 scenes ──
    for scene in SCENES:
        print(f"\n{'─' * 40}")
        print(f"Scene: {scene.upper()}")
        print(f"{'─' * 40}")

        for split in ["train", "val", "test"]:
            print(f"\n  [{split}]")
            dataset = get_ethucy_split(scene, split, data_root=DATA_ROOT, augment=False)


    print("\n" + "=" * 60)
    print("Example sample from ETH test set:")
    print("=" * 60)

    test_ds = get_ethucy_split("eth", "test", data_root=DATA_ROOT, augment=False)
    sample = test_ds[0]

    print(f"\n  Scene:    {sample['scene']}")
    print(f"  Ped ID:   {sample['ped_id']}")
    print(f"  Frame:    {sample['frame']}")
    print(f"\n  Observed positions (8 steps, 3.2s):")
    print(f"  {sample['observed']}")
    print(f"\n  Future positions (12 steps, 4.8s):")
    print(f"  {sample['future']}")
    print(f"\n  Shapes: observed={tuple(sample['observed'].shape)}, "
          f"future={tuple(sample['future'].shape)}")


    print("\n" + "=" * 60)
    print("DataLoader example:")
    print("=" * 60)

    train_ds = get_ethucy_split("eth", "train", data_root=DATA_ROOT, augment=True)
    loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)

    batch = next(iter(loader))
    print(f"\n  Batch observed shape: {tuple(batch['observed'].shape)}")
    print(f"  Batch future shape:   {tuple(batch['future'].shape)}")

ETH/UCY Dataset Loader

────────────────────────────────────────
Scene: ETH
────────────────────────────────────────

  [train]
  biwi_hotel_train.txt: 877 trajectory samples (311 pedestrians, 934 frames)
  crowds_zara01_train.txt: 1976 trajectory samples (125 pedestrians, 697 frames)
  crowds_zara02_train.txt: 4477 trajectory samples (171 pedestrians, 841 frames)
  crowds_zara03_train.txt: 1760 trajectory samples (106 pedestrians, 603 frames)
  students001_train.txt: 11691 trajectory samples (375 pedestrians, 355 frames)
  students003_train.txt: 8988 trajectory samples (364 pedestrians, 432 frames)
  uni_examples_train.txt: 538 trajectory samples (96 pedestrians, 587 frames)
  Total: 30307 samples for eth/train


  [val]
  biwi_hotel_val.txt: 318 trajectory samples (81 pedestrians, 234 frames)
  crowds_zara01_val.txt: 337 trajectory samples (29 pedestrians, 175 frames)
  crowds_zara02_val.txt: 1259 trajectory samples (47 pedestrians, 211 frames)
  crowds_zara03_val.txt: 708 trajectory